# Data Loading and Preprocessing - Experiment 3 Replication

This notebook handles the loading and preprocessing of Experiment 3 data from Rouault et al. (2019). It extracts the key data structures needed for subsequent analyses:

- **Exp3.mat**: Main MATLAB data file containing all experiment data
- **X_ser6_all**: Task choice data for Pairing 6 analysis (columns: trial_idx, pairing_id, difficulty, acc_diff, rt_diff, conf_diff, choice)

**Data Source**: https://github.com/marionrouault/RouaultDayanFleming

**Note**: This notebook is completely self-contained and runnable on any machine with the repository cloned.

In [35]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio

notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
project_root = notebook_dir.parent
data_file = project_root / "data" / "Exp3.mat"

print(f"Notebook location: {notebook_dir}")
print(f"Looking for data at: {data_file}")
print(f"Data file exists: {data_file.exists()}")

if not data_file.exists():
    raise FileNotFoundError(f"Could not find Exp3.mat at {data_file}. Please ensure the data file is in the correct location.")

mat = sio.loadmat(str(data_file), squeeze_me=True, struct_as_record=False)
Exp3 = mat["Exp3"]

print("Data loaded successfully")

Notebook location: /Users/ninaedgley/Library/CloudStorage/ProtonDrive-nina.edgley@protonmail.com-folder/rouault-exp3-replication-main/notebooks
Looking for data at: /Users/ninaedgley/Library/CloudStorage/ProtonDrive-nina.edgley@protonmail.com-folder/rouault-exp3-replication-main/data/Exp3.mat
Data file exists: True
Data loaded successfully


In [37]:
# Inspect Exp3.mat structure

print("Exp3.mat Structure")
print(f"Type: {type(Exp3)}")
print(f"Number of fields: {len([f for f in dir(Exp3) if not f.startswith('_')])}")

print("\nAvailable Fields")
fields = [f for f in dir(Exp3) if not f.startswith("_")]
for field in sorted(fields):
    try:
        value = getattr(Exp3, field)
        if hasattr(value, 'shape'):
            shape_info = f"shape={value.shape}"
        elif hasattr(value, '__len__') and not isinstance(value, str):
            shape_info = f"len={len(value)}"
        else:
            shape_info = f"type={type(value).__name__}"
        print(f"{field}: {shape_info}")
    except:
        print(f"{field}: <error accessing>")

print("\nKey Fields for Analysis")
key_fields = ['X_ser6_all', 'mratios', 'T1chperser', 'T2chperser', 'confidence_level']
for field in key_fields:
    if hasattr(Exp3, field):
        value = getattr(Exp3, field)
        print(f"{field}: {getattr(value, 'shape', 'no shape attribute')}")

Exp3.mat Structure
Type: <class 'scipy.io.matlab._mio5_params.mat_struct'>
Number of fields: 33

Available Fields
L_H1_H2: shape=(46, 8)
P: shape=(46, 4)
RT_H1_H2: shape=(46, 8)
RTt1perser: shape=(46, 6)
RTt2perser: shape=(46, 6)
T: shape=(46, 4)
T1chperser: shape=(46, 6)
T1chperserH: shape=(46, 6)
T1chperserL: shape=(46, 6)
T2chperser: shape=(46, 6)
T2chperserH: shape=(46, 6)
T2chperserL: shape=(46, 6)
X_ser6_all: shape=(230, 7)
acct1perser: shape=(46, 6)
acct2perser: shape=(46, 6)
auroc2: shape=(46, 2)
confCorrIncorr: shape=(46, 4)
conf_perf_correlation: shape=(46, 2)
confidence_level: shape=(46, 6)
confidence_level_pooled: shape=(46, 2)
evol_indiv_RT_LB: shape=(46, 8)
mratios: shape=(46, 2)
nR_S1_D: shape=(46,)
nR_S1_E: shape=(46,)
nR_S2_D: shape=(46,)
nR_S2_E: shape=(46,)
regLB_all: shape=(1380, 4)
task1val: shape=(6, 5, 46)
task2val: shape=(6, 5, 46)
task_ch_fbdif: shape=(46, 5)
task_ch_fbeasy: shape=(46, 5)
task_ch_nofbdif: shape=(46, 5)
task_ch_nofbeasy: shape=(46, 5)

Key Field

In [40]:
# Extract and Inspect X_ser6_all (task choice data)

X_ser6_all = Exp3.X_ser6_all
X = np.array(X_ser6_all, dtype=float)

print("X_ser6_all Overview")
print(f"Shape: {X.shape}")
print(f"Data type: {X.dtype}")
print(f"Memory usage: {X.nbytes} bytes")

columns = ["trial_idx", "pairing_id", "difficulty", "acc_diff", "rt_diff", "conf_diff", "choice"]
df = pd.DataFrame(X, columns=columns)

print("\nColumn Summary")
for col in df.columns:
    vals = df[col].values
    unique_vals = np.unique(vals)
    print(f"{col}: min={vals.min():.2f}, max={vals.max():.2f}, unique={len(unique_vals)}")

print("\nData Validation")
pairing_check = np.all(X[:, 1] == 6)
print(f"All pairing IDs = 6: {pairing_check}")
choice_check = set(np.unique(X[:, 6])).issubset({1, 2})
print(f"Choices are 1/2 only: {choice_check}")

print("\nChoice distribution")
choice_counts = df['choice'].value_counts().sort_index()
for choice, count in choice_counts.items():
    task = "Task 1" if choice == 1 else "Task 2"
    pct = count / len(df) * 100
    print(f"{task} (choice={int(choice)}): {count} trials ({pct:.1f}%)")

print("\nSample data (first 5 rows)")
print(df.head(5).round(4))

X_ser6_all Overview
Shape: (230, 7)
Data type: float64
Memory usage: 12880 bytes

Column Summary
trial_idx: min=0.00, max=29.00, unique=30
pairing_id: min=6.00, max=6.00, unique=1
difficulty: min=4.00, max=20.00, unique=5
acc_diff: min=-1.00, max=1.00, unique=54
rt_diff: min=-10846.80, max=429.30, unique=222
conf_diff: min=-0.29, max=0.50, unique=181
choice: min=1.00, max=2.00, unique=2

Data Validation
All pairing IDs = 6: True
Choices are 1/2 only: True

Choice distribution
Task 1 (choice=1): 74 trials (32.2%)
Task 2 (choice=2): 156 trials (67.8%)

Sample data (first 5 rows)
   trial_idx  pairing_id  difficulty  acc_diff   rt_diff  conf_diff  choice
0        8.0         6.0         4.0     -0.50 -489.5000     0.0900     2.0
1        9.0         6.0         8.0      0.25  145.0000     0.0025     2.0
2       12.0         6.0        16.0      0.25 -189.7500     0.0800     1.0
3       25.0         6.0        20.0     -0.10  -97.3000     0.0390     1.0
4       27.0         6.0        12.0